# z620 - LightGBM tweedie por cluster (Etapa D, consigna nueva)
`objective=tweedie`, `max_bin=1023`, Optuna, SOLO campos escalados como features. Predict en espacio escalado -> desescalar por `TN_promedio` de cada fila. Sin walk-forward (segun consigna, no vale la pena).

**ADVERTENCIA TECNICA (no resuelta, se deja explicita):** `objective=tweedie` de LightGBM requiere target &ge;0. El target `clase` es un DELTA (`clase_original_escalada - tn0_escalado`) y puede ser NEGATIVO cuando las ventas caen entre el ancla y el periodo objetivo. Esto es una contradiccion entre dos partes de la consigna (target=delta con signo, objetivo=tweedie que exige no-negatividad). Se implementa literal como se pidio; si LightGBM tira error o entrena mal por esto, hay que decidir explicitamente como resolverlo (no se decide aca por cuenta propia).

In [1]:
!pip install -q lightgbm pyarrow optuna polars

In [2]:
import os
import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb
import optuna
import warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [3]:
PARAM = {
    'experimento': 'LGB12_ESCALADO_CLUSTER',
    'kaggle_competition': 'labo-iii-2026-ba',
    'features_path': '/home/ds/exp/CP_CLUSTER_BRUTO/tb_FE_CP_con_cluster.parquet',
    'apredecir_path': '/home/ds/datasets/product_id_apredecir201912.txt',
    'periodo_ultimo_dato': 201912,
    'semilla': 102103,
    'n_trials': 30,          # por cluster -- bajar si son muchos clusters y el tiempo total importa
    'max_bin': 1023,
    'min_filas_por_cluster': 500
}

ruta = os.path.join('/home/ds/exp', PARAM['experimento'])
os.makedirs(ruta, exist_ok=True)
print(ruta)

/home/ds/exp/LGB12_ESCALADO_CLUSTER


In [4]:
df = pl.read_parquet(PARAM['features_path'])
print(df.shape)
print("minimo de 'clase' (chequeo de negatividad para tweedie):", df["clase"].min())

(16648066, 67)
minimo de 'clase' (chequeo de negatividad para tweedie): -33.99948411603997


## Definicion de features: SOLO campos escalados + categoricas de jerarquia
Se excluyen explicitamente los campos en magnitud real (`tn`, `tn0`, `tn1`, `TN_promedio`) y los que arman el target (`clase_original`, `clase_original_escalada`, `clase`).

In [5]:
cols_excluir = {
    "tn", "tn0", "tn1", "TN_promedio",
    "clase_original", "clase_original_escalada", "clase",
    "periodo", "periodo_target_m", "E_tn_shift1", "cluster_id"
}
features = [c for c in df.columns if c not in cols_excluir]
categoricas = [c for c in ["customer_id", "product_id", "cat1", "cat2", "cat3", "brand", "descripcion"] if c in features]
print(len(features), "features")
print(features)

56 features
['customer_id', 'product_id', 'tn0_escalado', 'E_tn', 'periodo_m', 'E_tn_lag_1', 'E_tn_lag_2', 'E_tn_lag_3', 'E_tn_lag_6', 'E_tn_lag_12', 'E_tn_delta_lag_1_2', 'E_tn_delta_lag_2_3', 'E_tn_delta_lag_3_6', 'E_tn_delta_lag_6_12', 'E_tn_media_3', 'E_tn_max_3', 'E_tn_min_3', 'E_tn_media_6', 'E_tn_max_6', 'E_tn_min_6', 'E_tn_media_9', 'E_tn_max_9', 'E_tn_min_9', 'E_tn_media_12', 'E_tn_max_12', 'E_tn_min_12', 'E_tn_media_18', 'E_tn_max_18', 'E_tn_min_18', 'E_tn_media_24', 'E_tn_max_24', 'E_tn_min_24', 'E_tn_media_36', 'E_tn_max_36', 'E_tn_min_36', 'E_tn_tendencia_3_6', 'E_tn_tendencia_6_9', 'E_tn_tendencia_9_12', 'E_tn_tendencia_12_18', 'E_tn_tendencia_18_24', 'E_tn_tendencia_24_36', 'ratio_E_tn_macro', 'cat1', 'cat2', 'cat3', 'brand', 'sku_size', 'descripcion', 'ratio_E_tn_cat1', 'ratio_E_tn_cat2', 'ratio_E_tn_cat3', 'ratio_E_tn_brand', 'ratio_E_tn_prod_todos_cli', 'ratio_E_tn_todos_tamanos', 'ratio_E_tn_muchos_cli', 'ratio_E_tn_complementarios']


## Split train / valid (train final 201910 -- SIN walk-forward, segun consigna)

In [6]:
def periodo_a_meses(periodo: int) -> int:
    return (periodo // 100) * 12 + (periodo % 100)

m_201910 = periodo_a_meses(201910)
m_201911 = periodo_a_meses(201911)
m_201912 = periodo_a_meses(201912)

def a_pandas(tabla):
    pdf = tabla.select(features + ["clase_original_escalada", "TN_promedio", "clase_original"]).to_pandas()
    for c in categoricas:
        pdf[c] = pdf[c].astype("category")
    return pdf

## Entrenamiento por cluster: Optuna (tweedie, max_bin=1023) + prediccion + desescalado

In [7]:
predicciones_totales = []
predicciones_validacion = []
clusters_saltados = []

for cluster_id in sorted(df["cluster_id"].unique().to_list()):
    sub = df.filter(pl.col("cluster_id") == cluster_id)

    sub_valido = sub.filter(pl.col("clase_original_escalada").is_not_null())
    train = sub_valido.filter(pl.col("periodo_target_m") <= m_201910)
    valid = sub_valido.filter(
        (pl.col("periodo_target_m") >= m_201911) & (pl.col("periodo_target_m") <= m_201912)
    )

    if train.height < PARAM['min_filas_por_cluster'] or valid.height < 20:
        print(f"cluster {cluster_id}: muy poca data (train={train.height}, valid={valid.height}), se salta")
        clusters_saltados.append(cluster_id)
        continue

    train_pd = a_pandas(train)
    valid_pd = a_pandas(valid)

    # target = clase_original_escalada directo (tn(p+2)/TN(p)) -- naturalmente >=0, sin desplazamiento artificial
    X_train = train_pd[features]
    y_train = train_pd["clase_original_escalada"]

    X_valid = valid_pd[features]
    y_valid = valid_pd["clase_original_escalada"]

    dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=categoricas,
                          params={'feature_pre_filter': False})
    dvalid = lgb.Dataset(X_valid, label=y_valid, categorical_feature=categoricas, reference=dtrain,
                          params={'feature_pre_filter': False})

    def objective(trial):
        params = {
            'objective': 'tweedie',
            'max_bin': PARAM['max_bin'],
            'verbosity': -1,
            'seed': PARAM['semilla'],
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 15, 255),
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 200),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
            'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        }
        modelo_trial = lgb.train(
            params, dtrain, num_boost_round=1000,
            valid_sets=[dvalid], callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )
        return modelo_trial.best_score['valid_0']['tweedie']

    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=PARAM['semilla']))
    study.optimize(objective, n_trials=PARAM['n_trials'], show_progress_bar=False)

    mejores_params = dict(study.best_params)
    mejores_params.update({'objective': 'tweedie', 'max_bin': PARAM['max_bin'], 'verbosity': -1, 'seed': PARAM['semilla']})

    modelo = lgb.train(
        mejores_params, dtrain, num_boost_round=1000,
        valid_sets=[dvalid], callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    futuro = sub.filter(pl.col("periodo") == PARAM['periodo_ultimo_dato'])
    futuro_pd = futuro.select(features).to_pandas()
    for c in categoricas:
        futuro_pd[c] = futuro_pd[c].astype("category")

    # prediccion DIRECTA de clase_original_escalada -- desescalado en un solo paso
    clase_original_escalada_pred = modelo.predict(futuro_pd, num_iteration=modelo.best_iteration)
    tn_promedio_futuro = futuro["TN_promedio"].to_numpy()

    clase_original_pred = clase_original_escalada_pred * tn_promedio_futuro
    clase_original_pred = np.clip(clase_original_pred, 0, None)

    res = futuro.select(["customer_id", "product_id"]).to_pandas()
    res["tn"] = clase_original_pred
    predicciones_totales.append(res)

    # prediccion tambien sobre VALIDACION, para el Total Error Rate
    valid_clase_original_escalada_pred = modelo.predict(X_valid, num_iteration=modelo.best_iteration)
    valid_clase_original_pred = valid_clase_original_escalada_pred * valid_pd["TN_promedio"].to_numpy()
    valid_clase_original_pred = np.clip(valid_clase_original_pred, 0, None)

    predicciones_validacion.append(pd.DataFrame({
        "pred": valid_clase_original_pred,
        "real": valid_pd["clase_original"].to_numpy()
    }))

    print(f"cluster {cluster_id}: train={train.height} valid={valid.height} mejor_tweedie={study.best_value:.4f}")

print("\nclusters entrenados:", len(predicciones_totales), " clusters saltados:", clusters_saltados)

cluster 0: train=1580442 valid=105966 mejor_tweedie=82.1862
cluster 1: train=1565585 valid=105966 mejor_tweedie=37.5317
cluster 2: train=1557693 valid=105966 mejor_tweedie=29.8258
cluster 3: train=1540901 valid=105966 mejor_tweedie=25.1968
cluster 4: train=1512780 valid=104139 mejor_tweedie=29.0040
cluster 5: train=1537213 valid=105966 mejor_tweedie=25.0071
cluster 6: train=1447232 valid=105966 mejor_tweedie=17.6557
cluster 7: train=1368885 valid=105966 mejor_tweedie=25.1014
cluster 8: train=1215116 valid=104158 mejor_tweedie=56.9490
cluster 9: train=901514 valid=86030 mejor_tweedie=31.9507

clusters entrenados: 10  clusters saltados: []


In [8]:
tabla_validacion = pd.concat(predicciones_validacion, ignore_index=True)

numerador = (tabla_validacion["pred"] - tabla_validacion["real"]).abs().sum()
denominador = tabla_validacion["real"].sum()
total_error_rate = numerador / denominador

print("Total Error Rate (validacion 201911-201912):", total_error_rate)

Total Error Rate (validacion 201911-201912): 2.4816822453349743


## Combinar, SUMAR por product_id, armar submit

In [9]:
resultado_cp = pd.concat(predicciones_totales, ignore_index=True)
resultado = resultado_cp.groupby("product_id", as_index=False)["tn"].sum()

apredecir = pl.read_csv(PARAM['apredecir_path'], separator="\t").to_pandas()
submit = apredecir[["product_id"]].merge(resultado, on="product_id", how="left")
print("nulos en submit (revisar):", submit["tn"].isna().sum())
submit["tn"] = submit["tn"].fillna(0.0)

archivo_submit = os.path.join(ruta, f"{PARAM['experimento']}_submit.csv")
submit.to_csv(archivo_submit, index=False)
print(archivo_submit)
submit.head()

nulos en submit (revisar): 0
/home/ds/exp/LGB12_ESCALADO_CLUSTER/LGB12_ESCALADO_CLUSTER_submit.csv


,product_id,tn
0,20001,2177.357759
1,20002,1549.760001
2,20003,1785.802702
3,20004,1027.191242
4,20005,1421.965554


In [10]:
def kaggle_submit(competencia, archivo, mensaje):
    comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
    os.system(comando)

kaggle_submit(PARAM['kaggle_competition'], archivo_submit, f"{PARAM['experimento']} escalado+cluster bruto+tweedie")

100%|██████████| 18.7k/18.7k [00:00<00:00, 60.6kB/s]


99 submissions remaining today.
Successfully submitted to Labo III, 2026 BA